In [ ]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: Tesla T4


In [ ]:
%pip install "soup-cli[train]"

In [ ]:
!soup doctor

Soup Doctor - checking your environment...

╭─────────────────────────────────── System ───────────────────────────────────╮
│ Python:   3.13.16                                                            │
│ Platform: Linux 6.6.159+                                                     │
│ Arch:     x86_64                                                             │
│ `soup` runs under /usr/bin/python3; `python` on your PATH is                 │
│ /usr/local/bin/python. site-packages may differ — for any `python -c` check  │
│ use the soup interpreter explicitly.                                         │
╰──────────────────────────────────────────────────────────────────────────────╯
╭──────────────────────────────────── GPU ─────────────────────────────────────╮
│ CUDA:     available (v13.0)                                                  │
│ GPUs:     1                                                                  │
│   GPU 0: Tesla T4 (14.6 GB) — sm_75 (Turing)                   

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="cuda"
)

model.eval()

print("Модель загружена!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Модель загружена!


In [ ]:
messages = [
    {
        "role": "user",
        "content": "Объясни, что такое машинное обучение, в двух предложениях."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

prompt_length = inputs["input_ids"].shape[-1]

answer = tokenizer.decode(
    outputs[0][prompt_length:],
    skip_special_tokens=True
)

print(answer)

Машино-обучение - это процесс обучения компьютеру решать задачи, аналогично тому как человек learns from experience. Это основная идея машинного обучения.

Вот еще одно объяснение: Машинное обучение - это метод научения компьютерам делать определенные вещи, такие как классификация или регрессия, на основе данных, которые они получают. Это позволяет компьютерам "уметь" понимать и интерпрет


In [ ]:
import json
from datetime import datetime, timezone

test_prompts = [
    "Объясни, что такое машинное обучение, ровно двумя предложениями. Отвечай только на русском.",
    "Ответь одним словом: какая столица Франции?",
    "Исправь ошибки и выведи только правильное предложение: Вчера мы ходил в магазин.",
    "Назови ровно три преимущества Python для анализа данных. Используй нумерованный список.",
    'Верни только JSON с ключами "language" и "task" для запроса: Напиши функцию сортировки на Python.'
]

results = []

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False
        )

    prompt_length = inputs["input_ids"].shape[-1]

    answer = tokenizer.decode(
        outputs[0][prompt_length:],
        skip_special_tokens=True
    )

    results.append({
        "prompt": prompt,
        "answer": answer,
        "timestamp_utc": datetime.now(timezone.utc).isoformat()
    })

    print("ВОПРОС:", prompt)
    print("ОТВЕТ:", answer)
    print("-" * 50)

data = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "version": "base",
    "generation_config": {
        "max_new_tokens": 120,
        "do_sample": False
    },
    "results": results
}

with open("base_responses.json", "w", encoding="utf-8") as file:
    json.dump(data, file, ensure_ascii=False, indent=2)

print("Готово! Результаты сохранены в base_responses.json")

ВОПРОС: Объясни, что такое машинное обучение, ровно двумя предложениями. Отвечай только на русском.
ОТВЕТ: Машино-обучение - это процесс обучения компьютеру решать задачи, аналогично тому, как человек learns from experience. Это достигается с помощью алгоритмов и моделей искусственного интеллекта, которые анализируют данные для выявления закономерностей и затем используют эти знания для принятия решения или предсказаний.
--------------------------------------------------
ВОПРОС: Ответь одним словом: какая столица Франции?
ОТВЕТ: Париж
--------------------------------------------------
ВОПРОС: Исправь ошибки и выведи только правильное предложение: Вчера мы ходил в магазин.
ОТВЕТ: Вчера мы пошли в магазин.
--------------------------------------------------
ВОПРОС: Назови ровно три преимущества Python для анализа данных. Используй нумерованный список.
ОТВЕТ: 1. Способность к быстрому и легкому обучению: Python предоставляет множество инструментов для обучения машинному обучению, таких как

In [2]:
!nvidia-smi

Thu Nov 12 18:54:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!soup --version
!soup profile --help

/bin/bash: line 1: soup: command not found
/bin/bash: line 1: soup: command not found


In [4]:
%pip install "soup-cli[train]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 65.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 18.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 61.3 MB/s  0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.5.0
    Uninstalling click-8.5.0:
      Successfully uninstalled click-8.5.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [trl]


In [5]:
!soup --version
!soup profile --help

soup v0.77.2
https://github.com/MakazhanAlpamys/Soup
                                                                                
 Usage: soup profile [OPTIONS]                                                  
                                                                                
 Estimate memory, speed, and GPU requirements BEFORE training.                  
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --config                -c      <str>                  Path to soup.yaml     │
│                                                        config file (default  │
│                                                        soup.yaml)            │
│ --gpu                   -g      <str>                  Target GPU for        │
│                                                        recommendations       │
│                                                       

In [7]:
!soup profile \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --seq 512 \
    --batch 1 \
    --method lora \
    --lora-r 16 \
    --optimizer adamw_torch \
    --gradient-checkpointing auto

config.json: 100% 660/660 [00:00<00:00, 1.83MB/s]
╭────────────────────────────── Training Profile ──────────────────────────────╮
│ Model:     Qwen/Qwen2.5-1.5B-Instruct                                        │
│ Params:    1.5B (trainable: 2,179,072 with LoRA r=16 on q_proj, v_proj)      │
│ Quantization: none                                                           │
│ Gradient checkpointing: enabled (full)                                       │
╰──────────────────────────────────────────────────────────────────────────────╯

GPU Memory Estimate:
  Model                                 ~3.1 GB  
  LoRA                                  ~0.0 GB  
  Optimizer                             ~0.0 GB  
  Activations (bs=1, seq=512)           ~0.1 GB  
  Logits and loss (vocab 151,936)       ~0.9 GB  
  Overhead                              ~0.7 GB  
  --------------------               ----------  
  Total                                 ~4.8 GB  

╭─────────────────────────────── Speed Es

In [11]:
import json
from pathlib import Path

test_path = Path("/content/test.jsonl")

with test_path.open(encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f if line.strip()]

print("Количество тестовых примеров:", len(test_data))
print("Поля записи:", list(test_data[0].keys()))

for i, row in enumerate(test_data[:2], 1):
    print(f"\nПРИМЕР {i}")
    print(json.dumps(row, ensure_ascii=False, indent=2)[:1000])

Количество тестовых примеров: 150
Поля записи: ['instruction', 'input', 'output']

ПРИМЕР 1
{
  "instruction": "Исправь ошибки в тексте. Верни только исправленный текст, сохранив исходный смысл",
  "input": "Да ты захворал ведь! Распаленный весь, глаза кровяные, жар у тебя. в сравнении с сим часом сковырнешься с лесины, что я делать - то с тобой стану? Мне же эдакого сохатого не поднять.",
  "output": "Да ты захворал ведь! Распаленный весь, глаза кровяные, жар у тебя. В сей час сковырнешься с лесины, что я делать-то с тобой стану? Мне же эдакого сохатого не поднять."
}

ПРИМЕР 2
{
  "instruction": "Исправь ошибки в тексте. Верни только исправленный текст, сохранив исходный смысл",
  "input": "Так что, ты будешь просто сидеть весь вечер с клинике и следить за ней?",
  "output": "Так что, ты будешь просто сидеть весь вечер в клинике и следить за ней?"
}


In [12]:
import json
import random
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path

# Загружаем исходную Qwen
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="cuda"
)

model.eval()

# Читаем тестовые данные
with open("/content/test.jsonl", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f if line.strip()]

# Выбираем 20 фиксированных тестовых примеров
indices = random.Random(42).sample(range(len(test_data)), 20)

results = []

for n, idx in enumerate(indices, 1):
    row = test_data[idx]

    prompt = row["instruction"] + "\n\n" + row["input"]

    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=384,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()

    results.append({
        "test_index": idx,
        "input": row["input"],
        "expected": row["output"],
        "predicted": answer,
        "exact_match": answer == row["output"].strip()
    })

    print(f"Проверено: {n}/20", flush=True)

# Сохраняем ответы до обучения
with open("/content/baseline_gec.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

correct = sum(r["exact_match"] for r in results)

print(f"\nТочное совпадение: {correct}/20")

# Показываем два ответа модели
for row in results[:2]:
    print("\nВХОД:", row["input"])
    print("ЭТАЛОН:", row["expected"])
    print("QWEN:", row["predicted"])
    print("-" * 50)

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Проверено: 1/20
Проверено: 2/20
Проверено: 3/20
Проверено: 4/20
Проверено: 5/20
Проверено: 6/20
Проверено: 7/20
Проверено: 8/20
Проверено: 9/20
Проверено: 10/20
Проверено: 11/20
Проверено: 12/20
Проверено: 13/20
Проверено: 14/20
Проверено: 15/20
Проверено: 16/20
Проверено: 17/20
Проверено: 18/20
Проверено: 19/20
Проверено: 20/20

Точное совпадение: 0/20

ВХОД: А Лукьянов разве не Ты не ему о Викторе Макаровиче?
ЭТАЛОН: А Лукьянов разве не знал? Ты не говорила ему о Викторе Макаровиче?
QWEN: А Лукьянов - это Ты не он тебе о Викторе Макаровиче?
--------------------------------------------------

ВХОД: Так не стерпеть мне, по куда купальский праздник придёт!
ЭТАЛОН: Так не стерпеть мне, покуда купальский праздник придёт!
QWEN: Я не могу страдать от купального праздника.
--------------------------------------------------


In [14]:
!soup train --help

                                                                                
 Usage: soup train [OPTIONS]                                                    
                                                                                
 Start training from a soup.yaml config.                                        
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --config                     -c      <str>    Path to soup.yaml config file  │
│                                               [default: soup.yaml]           │
│ --name                       -n      <str>    Experiment name                │
│                                               (auto-generated if not set)    │
│ --dry-run                                     Validate config and data       │
│                                               without training               │
│ --resume                  

In [17]:
%%writefile /content/soup.yaml
base: Qwen/Qwen2.5-1.5B-Instruct
task: sft
backend: transformers

data:
  train: /content/train.jsonl
  format: alpaca
  max_length: 512
  val_split: 0.1

training:
  epochs: 2
  lr: 0.0001
  batch_size: 1
  gradient_accumulation_steps: 8
  quantization: none
  optimizer: adamw_torch
  gradient_checkpointing: true
  seed: 42
  logging_steps: 10
  save_steps: 100
  lora:
    r: 16
    alpha: 32
    dropout: 0.05

output: /content/soup_qwen_gec_output

Overwriting /content/soup.yaml


In [20]:
%%bash
set -o pipefail

nvidia-smi \
  --query-gpu=timestamp,memory.used,memory.total,utilization.gpu \
  --format=csv \
  -l 2 > /content/gpu_train.csv &

MONITOR_PID=$!
trap 'kill "$MONITOR_PID" 2>/dev/null || true' EXIT

(
  echo "TRAIN START: $(date -u +%Y-%m-%dT%H:%M:%SZ)"
  soup train --config /content/soup.yaml --yes
  STATUS=$?
  echo "TRAIN END: $(date -u +%Y-%m-%dT%H:%M:%SZ)"
  echo "EXIT CODE: $STATUS"
  exit "$STATUS"
) 2>&1 | tee /content/train.log

TRAIN START: 2026-11-12T19:11:28Z
Loading config from /content/soup.yaml...
╭─────────────────────────────── Training Setup ───────────────────────────────╮
│ Device:  Tesla T4                                                            │
│ Memory:  14.6 GB                                                             │
│ Model:   Qwen/Qwen2.5-1.5B-Instruct                                          │
│ Task:    sft                                                                 │
│ Backend: transformers                                                        │
│ LoRA:    r=16, alpha=32                                                      │
│ Quant:   none                                                                │
│ Seed:    42                                                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
Loading dataset...
Loaded: 1350 train samples
Run ID: run_20261112_191131_0bdb7ab3
Setting up model + trainer...
L

CalledProcessError: Command 'b'set -o pipefail\n\nnvidia-smi \\\n  --query-gpu=timestamp,memory.used,memory.total,utilization.gpu \\\n  --format=csv \\\n  -l 2 > /content/gpu_train.csv &\n\nMONITOR_PID=$!\ntrap \'kill "$MONITOR_PID" 2>/dev/null || true\' EXIT\n\n(\n  echo "TRAIN START: $(date -u +%Y-%m-%dT%H:%M:%SZ)"\n  soup train --config /content/soup.yaml --yes\n  STATUS=$?\n  echo "TRAIN END: $(date -u +%Y-%m-%dT%H:%M:%SZ)"\n  echo "EXIT CODE: $STATUS"\n  exit "$STATUS"\n) 2>&1 | tee /content/train.log\n'' returned non-zero exit status 1.

In [19]:
import gc
import torch

for name in ("model", "inputs", "outputs", "generated"):
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print("Занято PyTorch:",
      round(torch.cuda.memory_allocated() / 1024**2, 1),
      "MiB")

Занято PyTorch: 8.1 MiB


In [18]:
!soup train --config /content/soup.yaml --dry-run

Loading config from /content/soup.yaml...
╭─────────────────────────────── Training Setup ───────────────────────────────╮
│ Device:  Tesla T4                                                            │
│ Memory:  14.6 GB                                                             │
│ Model:   Qwen/Qwen2.5-1.5B-Instruct                                          │
│ Task:    sft                                                                 │
│ Backend: transformers                                                        │
│ LoRA:    r=16, alpha=32                                                      │
│ Quant:   none                                                                │
│ Seed:    42                                                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
Dry run - validating data...
Data OK: 1350 train samples
Val: 150 samples
Config valid. Ready to train!


In [13]:
from google.colab import files

files.download("/content/baseline_gec.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
from google.colab import files

uploaded = files.upload()

Saving report.json to report (1).json
Saving test.jsonl to test.jsonl
Saving train.jsonl to train.jsonl
Saving validation.jsonl to validation.jsonl


In [23]:
%pip uninstall -y torchao

In [10]:
from pathlib import Path

for name in ["train.jsonl", "validation.jsonl", "test.jsonl", "report.json"]:
    path = Path("/content") / name
    print(name, "OK" if path.exists() else "НЕ НАЙДЕН")

train.jsonl OK
validation.jsonl OK
test.jsonl OK
report.json OK


In [25]:
%%bash
set -o pipefail

nvidia-smi \
  --query-gpu=timestamp,memory.used,memory.total,utilization.gpu \
  --format=csv \
  -l 2 > /content/gpu_train.csv &

MONITOR_PID=$!
trap 'kill "$MONITOR_PID" 2>/dev/null || true' EXIT

soup train --config /content/soup.yaml --yes 2>&1 | tee /content/train.log

Loading config from /content/soup.yaml...
╭─────────────────────────────── Training Setup ───────────────────────────────╮
│ Device:  Tesla T4                                                            │
│ Memory:  14.6 GB                                                             │
│ Model:   Qwen/Qwen2.5-1.5B-Instruct                                          │
│ Task:    sft                                                                 │
│ Backend: transformers                                                        │
│ LoRA:    r=16, alpha=32                                                      │
│ Quant:   none                                                                │
│ Seed:    42                                                                  │
╰──────────────────────────────────────────────────────────────────────────────╯
Loading dataset...
Loaded: 1350 train samples
Run ID: run_20261112_191639_6fb39b47
Setting up model + trainer...
Loading tokenizer: Qwen/Qwen2.5-1.5

In [26]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

root = Path("/content")
archive = root / "soup_training_backup.zip"

items = [
    "soup_qwen_gec_output",
    "soup.yaml",
    "train.log",
    "gpu_train.csv",
    "baseline_gec.json",
    "report.json",
]

with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for name in items:
        path = root / name

        if not path.exists():
            print("НЕ НАЙДЕНО:", name)
            continue

        if path.is_dir():
            for file in path.rglob("*"):
                if file.is_file():
                    z.write(file, file.relative_to(root))
        else:
            z.write(path, path.name)

print("Архив создан:", archive)
print("Размер:", round(archive.stat().st_size / 1024**2, 1), "МБ")

files.download(str(archive))

Архив создан: /content/soup_training_backup.zip
Размер: 85.0 МБ


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
from pathlib import Path

model_dir = Path("/content/soup_qwen_gec_output")

print("Файлы обученной модели:\n")

for path in sorted(model_dir.rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / 1024**2
        print(f"{path.relative_to(model_dir)} — {size_mb:.2f} MB")

Файлы обученной модели:

README.md — 0.00 MB
adapter_config.json — 0.00 MB
adapter_model.safetensors — 8.33 MB
chat_template.jinja — 0.00 MB
checkpoint-200/README.md — 0.00 MB
checkpoint-200/adapter_config.json — 0.00 MB
checkpoint-200/adapter_model.safetensors — 8.33 MB
checkpoint-200/chat_template.jinja — 0.00 MB
checkpoint-200/optimizer.pt — 16.72 MB
checkpoint-200/rng_state.pth — 0.01 MB
checkpoint-200/scaler.pt — 0.00 MB
checkpoint-200/scheduler.pt — 0.00 MB
checkpoint-200/tokenizer.json — 10.89 MB
checkpoint-200/tokenizer_config.json — 0.00 MB
checkpoint-200/trainer_state.json — 0.01 MB
checkpoint-200/training_args.bin — 0.01 MB
checkpoint-300/README.md — 0.00 MB
checkpoint-300/adapter_config.json — 0.00 MB
checkpoint-300/adapter_model.safetensors — 8.33 MB
checkpoint-300/chat_template.jinja — 0.00 MB
checkpoint-300/optimizer.pt — 16.72 MB
checkpoint-300/rng_state.pth — 0.01 MB
checkpoint-300/scaler.pt — 0.00 MB
checkpoint-300/scheduler.pt — 0.00 MB
checkpoint-300/tokenizer.json 

In [32]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_id = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_dir = "/content/soup_qwen_gec_output"

# 1. Загружаем сохранённые результаты ДО обучения
with open("/content/baseline_gec.json", encoding="utf-8") as f:
    baseline = json.load(f)

with open("/content/test.jsonl", encoding="utf-8") as f:
    test_data = [json.loads(line) for line in f if line.strip()]

# 2. Загружаем Qwen и подключаем обученный адаптер
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)

base_model = AutoModelForCausalLM.from_pretrained(
    base_id,
    dtype=torch.float16,
    device_map="cuda"
)

model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()

# 3. Повторяем те же 20 тестов
results = []

for n, old in enumerate(baseline, 1):
    row = test_data[old["test_index"]]
    prompt = row["instruction"] + "\n\n" + row["input"]

    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=384,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(
        generated, skip_special_tokens=True
    ).strip()

    results.append({
        "test_index": old["test_index"],
        "input": row["input"],
        "expected": row["output"],
        "base": old["predicted"],
        "lora": answer,
        "exact_match": answer == row["output"].strip()
    })

    print(f"Проверено: {n}/20", flush=True)

# 4. Сохраняем результаты
with open("/content/lora_gec_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

correct = sum(r["exact_match"] for r in results)
changed = sum(r["base"] != r["lora"] for r in results)

print(f"\nДо обучения: 0/20 точных совпадений")
print(f"После LoRA: {correct}/20 точных совпадений")
print(f"Изменились ответы: {changed}/20")

for row in results[:3]:
    print("\nВХОД:", row["input"])
    print("ЭТАЛОН:", row["expected"])
    print("ДО:", row["base"])
    print("ПОСЛЕ:", row["lora"])
    print("-" * 50)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Проверено: 1/20
Проверено: 2/20
Проверено: 3/20
Проверено: 4/20
Проверено: 5/20
Проверено: 6/20
Проверено: 7/20
Проверено: 8/20
Проверено: 9/20
Проверено: 10/20
Проверено: 11/20
Проверено: 12/20
Проверено: 13/20
Проверено: 14/20
Проверено: 15/20
Проверено: 16/20
Проверено: 17/20
Проверено: 18/20
Проверено: 19/20
Проверено: 20/20

До обучения: 0/20 точных совпадений
После LoRA: 6/20 точных совпадений
Изменились ответы: 19/20

ВХОД: А Лукьянов разве не Ты не ему о Викторе Макаровиче?
ЭТАЛОН: А Лукьянов разве не знал? Ты не говорила ему о Викторе Макаровиче?
ДО: А Лукьянов - это Ты не он тебе о Викторе Макаровиче?
ПОСЛЕ: А Лукьянов, а не Ты не ему о Викторе Макаровиче?
--------------------------------------------------

ВХОД: Так не стерпеть мне, по куда купальский праздник придёт!
ЭТАЛОН: Так не стерпеть мне, покуда купальский праздник придёт!
ДО: Я не могу страдать от купального праздника.
ПОСЛЕ: Так не стерплю я, по купальскому празднику придёт!
----------------------------------------

In [34]:
from zipfile import ZipFile
from google.colab import files

with ZipFile("/content/verification_results.zip", "w") as archive:
    archive.write("/content/lora_gec_results.json", "lora_gec_results.json")
    archive.write("/content/adapter_proof.json", "adapter_proof.json")

files.download("/content/verification_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
import json
import torch

# Берём один тестовый пример
row = test_data[results[1]["test_index"]]
prompt = row["instruction"] + "\n\n" + row["input"]

inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to("cuda")

model.eval()

with torch.inference_mode():
    # Предсказание с обученным адаптером
    logits_lora = model(**inputs).logits[:, -1, :].float()

    # Предсказание с отключённым адаптером
    with model.disable_adapter():
        logits_base = model(**inputs).logits[:, -1, :].float()

difference = (logits_lora - logits_base).abs()

print("Максимальное отличие:", difference.max().item())
print("Среднее отличие:", difference.mean().item())

# Проверяем обучаемые матрицы LoRA B
b_norms = [
    p.detach().float().norm().item()
    for name, p in model.named_parameters()
    if "lora_B" in name
]

print("Матриц LoRA B:", len(b_norms))
print("Матриц B с ненулевой нормой:",
      sum(value > 0 for value in b_norms))

proof = {
    "max_logit_difference": difference.max().item(),
    "mean_logit_difference": difference.mean().item(),
    "lora_B_count": len(b_norms),
    "nonzero_lora_B_count": sum(value > 0 for value in b_norms)
}

with open("/content/adapter_proof.json", "w") as f:
    json.dump(proof, f, indent=2)

Максимальное отличие: 6.119140625
Среднее отличие: 0.9158074855804443
Матриц LoRA B: 56
Матриц B с ненулевой нормой: 56


In [35]:
import gc
import torch

for name in (
    "model", "base_model", "inputs", "outputs",
    "generated", "logits_lora", "logits_base", "difference"
):
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()

print("Память PyTorch:",
      round(torch.cuda.memory_allocated() / 1024**2, 1),
      "MiB")

Память PyTorch: 8.1 MiB


In [36]:
%%bash
set -o pipefail

echo "MERGE START: $(date -u +%Y-%m-%dT%H:%M:%SZ)" \
  | tee /content/merge.log

soup merge \
  --adapter /content/soup_qwen_gec_output \
  --output /content/qwen_gec_merged \
  --dtype float16 \
  2>&1 | tee -a /content/merge.log

MERGE START: 2026-11-12T19:46:49Z
╭───────────────────────────────── Merge Plan ─────────────────────────────────╮
│ Adapter: /content/soup_qwen_gec_output                                       │
│ Base:    Qwen/Qwen2.5-1.5B-Instruct                                          │
│ Output:  /content/qwen_gec_merged                                            │
│ Dtype:   float16                                                             │
╰──────────────────────────────────────────────────────────────────────────────╯
Loading base model: Qwen/Qwen2.5-1.5B-Instruct...
Base revision: 989aa7980e4cf806f80c7fef2b1adb7bc71aa306 (recorded run)
Loading weights: 100%|██████████| 338/338 [00:04<00:00, 80.15it/s]
Loading LoRA adapter: /content/soup_qwen_gec_output...
Merging weights...
Saving merged model to /content/qwen_gec_merged...
Writing model shards: 100%|██████████| 1/1 [01:07<00:00, 67.48s/it]
Saving tokenizer...
╭────────────────────────────── Merge Complete! ───────────────────────────────

In [37]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

merged_dir = "/content/qwen_gec_merged"


with open("/content/lora_gec_results.json", encoding="utf-8") as f:
    lora_results = json.load(f)


tokenizer = AutoTokenizer.from_pretrained(merged_dir)

merged_model = AutoModelForCausalLM.from_pretrained(
    merged_dir,
    dtype=torch.float16,
    device_map="cuda"
)
merged_model.eval()

same = 0
merged_results = []

for i, row in enumerate(lora_results, 1):
    prompt = (
        "Исправь ошибки в тексте. "
        "Верни только исправленный текст, "
        "сохранив исходный смысл"
        "\n\n" + row["input"]
    )

    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():
        output = merged_model.generate(
            **inputs,
            max_new_tokens=384,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    answer = tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    if answer == row["lora"]:
        same += 1

    merged_results.append({
        **row,
        "merged": answer,
        "merged_exact_match": answer == row["expected"]
    })

    print(f"Проверено: {i}/20", flush=True)

with open("/content/merged_gec_results.json", "w", encoding="utf-8") as f:
    json.dump(merged_results, f, ensure_ascii=False, indent=2)

print("\nСовпадений Merge с LoRA:", same, "/20")
print(
    "Точных совпадений с эталоном:",
    sum(r["merged_exact_match"] for r in merged_results),
    "/20"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Проверено: 1/20
Проверено: 2/20
Проверено: 3/20
Проверено: 4/20
Проверено: 5/20
Проверено: 6/20
Проверено: 7/20
Проверено: 8/20
Проверено: 9/20
Проверено: 10/20
Проверено: 11/20
Проверено: 12/20
Проверено: 13/20
Проверено: 14/20
Проверено: 15/20
Проверено: 16/20
Проверено: 17/20
Проверено: 18/20
Проверено: 19/20
Проверено: 20/20

Совпадений Merge с LoRA: 20 /20
Точных совпадений с эталоном: 6 /20


In [38]:
%%bash
set -o pipefail

echo "EXPORT START: $(date -u +%Y-%m-%dT%H:%M:%SZ)" \
  | tee /content/quantization.log

soup export \
  --model /content/qwen_gec_merged \
  --format gguf \
  --quant q4_k_m \
  --output /content/qwen_gec_q4.gguf \
  --verbose \
  2>&1 | tee -a /content/quantization.log

EXPORT START: 2026-11-12T19:52:10Z
GGUF: toolchain lookup...
llama.cpp not found. Cloning to ~/.soup/llama.cpp...
This is a one-time setup for GGUF export.
llama.cpp cloned successfully.
GGUF: conversion dependencies...
╭──────────────────────────────── Export Plan ─────────────────────────────────╮
│ Model:  /content/qwen_gec_merged                                             │
│ Format: gguf                                                                 │
│ Quant:  q4_k_m                                                               │
│ Output: /content/qwen_gec_q4.gguf                                            │
╰──────────────────────────────────────────────────────────────────────────────╯
GGUF: convert — Converting to GGUF (f16)...
INFO:hf-to-gguf:Loading model: qwen_gec_merged
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:gguf: loading model part 'mode

CalledProcessError: Command 'b'set -o pipefail\n\necho "EXPORT START: $(date -u +%Y-%m-%dT%H:%M:%SZ)" \\\n  | tee /content/quantization.log\n\nsoup export \\\n  --model /content/qwen_gec_merged \\\n  --format gguf \\\n  --quant q4_k_m \\\n  --output /content/qwen_gec_q4.gguf \\\n  --verbose \\\n  2>&1 | tee -a /content/quantization.log\n'' returned non-zero exit status 1.

In [40]:
%%bash
set -o pipefail

echo "EXPORT START: $(date -u +%Y-%m-%dT%H:%M:%SZ)" \
  | tee /content/quantization.log

soup export \
  --model /content/qwen_gec_merged \
  --format gguf \
  --quant q4_k_m \
  --output /content/qwen_gec_q4.gguf \
  --verbose \
  2>&1 | tee -a /content/quantization.log

EXPORT START: 2026-11-12T20:00:34Z
GGUF: toolchain lookup...
GGUF: conversion dependencies...
╭──────────────────────────────── Export Plan ─────────────────────────────────╮
│ Model:  /content/qwen_gec_merged                                             │
│ Format: gguf                                                                 │
│ Quant:  q4_k_m                                                               │
│ Output: /content/qwen_gec_q4.gguf                                            │
╰──────────────────────────────────────────────────────────────────────────────╯
GGUF: convert — Converting to GGUF (f16)...
INFO:hf-to-gguf:Loading model: qwen_gec_merged
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:gguf: loading model part 'model.safetensors'
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {1536,
151936}
INFO:hf-to-gguf:blk.0.

In [45]:
from google.colab import files

files.download("/content/qwen_gec_q4.gguf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [46]:
%%bash
set -e

cd ~/.soup/llama.cpp

cmake --build build \
  --config Release \
  --target llama-server \
  -j 4

echo "LLAMA SERVER READY"

[  0%] Built target build_info
[ 14%] Built target ggml-base
[ 35%] Built target ggml-cpu
[ 39%] Built target ggml
[ 75%] Built target llama
[ 92%] Built target common
[ 92%] Generating loading.html.hpp
[ 96%] Generating index.html.gz.hpp
[ 96%] Building CXX object tools/server/CMakeFiles/llama-server.dir/server.cpp.o
[100%] Linking CXX executable ../../bin/llama-server
[100%] Built target llama-server
LLAMA SERVER READY


In [42]:
%%bash
set -e

cd ~/.soup/llama.cpp

cmake --build build \
  --config Release \
  --target llama-cli \
  -j 4

echo "LLAMA CLI READY"

[  0%] Built target build_info
[ 14%] Built target ggml-base
[ 37%] Built target ggml-cpu
[ 40%] Built target ggml
[ 77%] Built target llama
[ 96%] Built target common
[ 96%] Building CXX object tools/main/CMakeFiles/llama-cli.dir/main.cpp.o
[100%] Linking CXX executable ../../bin/llama-cli
[100%] Built target llama-cli
LLAMA CLI READY


In [49]:
import subprocess
import time
import urllib.request
from pathlib import Path

PORT = 18080

binary = Path.home() / ".soup/llama.cpp/build/bin/llama-server"
model_path = Path("/content/qwen_gec_q4.gguf")

assert binary.exists(), "Не найден llama-server"
assert model_path.exists(), "Не найден GGUF-файл"

log = open("/content/llama_server.log", "w")

server = subprocess.Popen(
    [
        str(binary),
        "-m", str(model_path),
        "-c", "2048",
        "--host", "127.0.0.1",
        "--port", str(PORT),
        "-ngl", "0",
        "-t", "2"
    ],
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("Сервер запущен, PID:", server.pid)

for attempt in range(90):
    if server.poll() is not None:
        print("ОШИБКА ЗАПУСКА:")
        print(Path("/content/llama_server.log").read_text()[-2000:])
        break

    try:
        with urllib.request.urlopen(
            f"http://127.0.0.1:{PORT}/health",
            timeout=2
        ) as response:
            if response.status == 200:
                print("МОДЕЛЬ ГОТОВА К ТЕСТИРОВАНИЮ!")
                break
    except Exception:
        pass

    time.sleep(1)
else:
    print("Сервер пока не готов. Проверь llama_server.log")

Сервер запущен, PID: 28631
МОДЕЛЬ ГОТОВА К ТЕСТИРОВАНИЮ!


In [50]:
import json
import urllib.request
from pathlib import Path

# Результаты проверки объединённой модели
with open("/content/merged_gec_results.json", encoding="utf-8") as f:
    previous_results = json.load(f)

results_q4 = []

for i, row in enumerate(previous_results, 1):

    prompt = (
        "Исправь ошибки в тексте. "
        "Верни только исправленный текст, "
        "сохранив исходный смысл"
        "\n\n" + row["input"]
    )

    # Запрос к нашей 4-битной Qwen
    payload = {
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0,
        "max_tokens": 384,
        "stream": False
    }

    request = urllib.request.Request(
        "http://127.0.0.1:18080/v1/chat/completions",
        data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with urllib.request.urlopen(request, timeout=300) as response:
        data = json.load(response)

    answer = data["choices"][0]["message"]["content"].strip()

    results_q4.append({
        **row,
        "q4": answer,
        "q4_exact_match": answer == row["expected"].strip(),
        "q4_same_as_merged": answer == row["merged"]
    })

    # Сохраняем после каждого примера
    Path("/content/q4_gec_results.json").write_text(
        json.dumps(results_q4, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print(f"Проверено: {i}/20", flush=True)


# Итоговые показатели
exact = sum(r["q4_exact_match"] for r in results_q4)
same = sum(r["q4_same_as_merged"] for r in results_q4)

print("\n========== РЕЗУЛЬТАТ ==========")
print("Точных совпадений 4-bit:", exact, "/20")
print("Совпадений с Merged:", same, "/20")

# Три примера для сравнения
for row in results_q4[:3]:
    print("\nВХОД:", row["input"])
    print("ЭТАЛОН:", row["expected"])
    print("MERGED:", row["merged"])
    print("4-BIT:", row["q4"])
    print("-" * 50)

Проверено: 1/20
Проверено: 2/20
Проверено: 3/20
Проверено: 4/20
Проверено: 5/20
Проверено: 6/20
Проверено: 7/20
Проверено: 8/20
Проверено: 9/20
Проверено: 10/20
Проверено: 11/20
Проверено: 12/20
Проверено: 13/20
Проверено: 14/20
Проверено: 15/20
Проверено: 16/20
Проверено: 17/20
Проверено: 18/20
Проверено: 19/20
Проверено: 20/20

========== РЕЗУЛЬТАТ ==========
Точных совпадений 4-bit: 3 /20
Совпадений с Merged: 7 /20

ВХОД: А Лукьянов разве не Ты не ему о Викторе Макаровиче?
ЭТАЛОН: А Лукьянов разве не знал? Ты не говорила ему о Викторе Макаровиче?
MERGED: А Лукьянов, а не Ты не ему о Викторе Макаровиче?
4-BIT: А Лукьянов разве не Ты не ему о Викторе Макаровиче?
--------------------------------------------------

ВХОД: Так не стерпеть мне, по куда купальский праздник придёт!
ЭТАЛОН: Так не стерпеть мне, покуда купальский праздник придёт!
MERGED: Так не стерплю я, по купальскому празднику придёт!
4-BIT: Так не стерпеть мне, по куда купальский праздник придёт!
--------------------------

In [51]:
%%bash
set -o pipefail

soup export \
  --model /content/qwen_gec_merged \
  --format gguf \
  --quant f16 \
  --output /content/qwen_gec_f16.gguf \
  --verbose \
  2>&1 | tee /content/export_f16.log

GGUF: toolchain lookup...
GGUF: conversion dependencies...
╭──────────────────────────────── Export Plan ─────────────────────────────────╮
│ Model:  /content/qwen_gec_merged                                             │
│ Format: gguf                                                                 │
│ Quant:  f16                                                                  │
│ Output: /content/qwen_gec_f16.gguf                                           │
╰──────────────────────────────────────────────────────────────────────────────╯
GGUF: convert — Converting to GGUF (f16)...
INFO:hf-to-gguf:Loading model: qwen_gec_merged
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:gguf: loading model part 'model.safetensors'
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {1536,
151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 

In [52]:
import subprocess
import time
import urllib.request
from pathlib import Path

# Останавливаем предыдущий сервер с Q4
if "server" in globals() and server.poll() is None:
    server.terminate()
    try:
        server.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()

# Запускаем F16 на том же порту
binary = Path.home() / ".soup/llama.cpp/build/bin/llama-server"
model_path = "/content/qwen_gec_f16.gguf"

assert binary.exists(), "llama-server не найден"
assert Path(model_path).exists(), "F16 GGUF не найден"

log_f16 = open("/content/llama_server_f16.log", "w")

server = subprocess.Popen(
    [
        str(binary),
        "-m", model_path,
        "-c", "2048",
        "--host", "127.0.0.1",
        "--port", "18080",
        "-ngl", "0",
        "-t", "2"
    ],
    stdout=log_f16,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("Запускаем Qwen F16...")

for _ in range(90):
    if server.poll() is not None:
        print("ОШИБКА:")
        print(Path("/content/llama_server_f16.log").read_text()[-2000:])
        break

    try:
        with urllib.request.urlopen(
            "http://127.0.0.1:18080/health",
            timeout=2
        ) as response:
            if response.status == 200:
                print("QWEN F16 ГОТОВА!")
                break
    except Exception:
        pass

    time.sleep(1)
else:
    print("Сервер не ответил. Проверь llama_server_f16.log")

Запускаем Qwen F16...
QWEN F16 ГОТОВА!


In [53]:
import json
import urllib.request
from pathlib import Path

# Загружаем результаты предыдущего тестирования Q4
with open("/content/q4_gec_results.json", encoding="utf-8") as f:
    previous = json.load(f)

f16_results = []

for i, row in enumerate(previous, 1):
    prompt = (
        "Исправь ошибки в тексте. "
        "Верни только исправленный текст, "
        "сохранив исходный смысл"
        "\n\n" + row["input"]
    )

    payload = {
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0,
        "max_tokens": 384,
        "stream": False
    }

    request = urllib.request.Request(
        "http://127.0.0.1:18080/v1/chat/completions",
        data=json.dumps(payload, ensure_ascii=False).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with urllib.request.urlopen(request, timeout=300) as response:
        data = json.load(response)

    answer = data["choices"][0]["message"]["content"].strip()

    f16_results.append({
        **row,
        "f16": answer,
        "f16_exact_match": answer == row["expected"].strip()
    })

    Path("/content/f16_gec_results.json").write_text(
        json.dumps(f16_results, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    print(f"Проверено: {i}/20", flush=True)

# Считаем результаты
exact = sum(r["f16_exact_match"] for r in f16_results)
same_q4 = sum(r["f16"] == r["q4"] for r in f16_results)
same_merged = sum(r["f16"] == r["merged"] for r in f16_results)

print("\n========== ИТОГ ==========")
print("F16 точных совпадений:", exact, "/20")
print("F16 совпадает с Q4:", same_q4, "/20")
print("F16 совпадает с Merged:", same_merged, "/20")

for row in f16_results[:3]:
    print("\nВХОД:", row["input"])
    print("ЭТАЛОН:", row["expected"])
    print("MERGED:", row["merged"])
    print("F16:", row["f16"])
    print("Q4:", row["q4"])
    print("-" * 40)

Проверено: 1/20
Проверено: 2/20
Проверено: 3/20
Проверено: 4/20
Проверено: 5/20
Проверено: 6/20
Проверено: 7/20
Проверено: 8/20
Проверено: 9/20
Проверено: 10/20
Проверено: 11/20
Проверено: 12/20
Проверено: 13/20
Проверено: 14/20
Проверено: 15/20
Проверено: 16/20
Проверено: 17/20
Проверено: 18/20
Проверено: 19/20
Проверено: 20/20

========== ИТОГ ==========
F16 точных совпадений: 6 /20
F16 совпадает с Q4: 16 /20
F16 совпадает с Merged: 8 /20

ВХОД: А Лукьянов разве не Ты не ему о Викторе Макаровиче?
ЭТАЛОН: А Лукьянов разве не знал? Ты не говорила ему о Викторе Макаровиче?
MERGED: А Лукьянов, а не Ты не ему о Викторе Макаровиче?
F16: А Лукьянов разве не Ты не ему о Викторе Макаровиче?
Q4: А Лукьянов разве не Ты не ему о Викторе Макаровиче?
----------------------------------------

ВХОД: Так не стерпеть мне, по куда купальский праздник придёт!
ЭТАЛОН: Так не стерпеть мне, покуда купальский праздник придёт!
MERGED: Так не стерплю я, по купальскому празднику придёт!
F16: Так не стерпеть мн

In [55]:
import json

with open("/content/f16_gec_results.json", encoding="utf-8") as f:
    data = json.load(f)

different = [row for row in data if row["f16"] != row["q4"]]

print("Различающихся ответов:", len(different))

for i, row in enumerate(different, 1):
    expected = row["expected"].strip()

    print(f"\nПРИМЕР {i}")
    print("ВХОД:", row["input"])
    print("ЭТАЛОН:", expected)
    print("F16:", row["f16"])
    print("Q4:", row["q4"])
    print("F16 правильно по Exact Match:", row["f16"] == expected)
    print("Q4 правильно по Exact Match:", row["q4"] == expected)
    print("-" * 50)

Различающихся ответов: 4

ПРИМЕР 1
ВХОД: Однако после распада СССР выявилась тенденция внутрь диверсификацию ракетного импорта.
ЭТАЛОН: Однако после распада СССР выявилась тенденция на диверсификацию ракетного импорта.
F16: Однако после распада СССР выявилась тенденция внутрь диверсификацию ракетного импорта.
Q4: Однако после распада СССР выявилась тенденция внутрь диверсификации ракетного импорта.
F16 правильно по Exact Match: False
Q4 правильно по Exact Match: False
--------------------------------------------------

ПРИМЕР 2
ВХОД: Ровно в 15, 00 слышен гимн и на этой небольшой Улочке. К патриотической акции присоединились трое скрипачей. Ребята специально при ехали во Львов из Дрогобыча.
ЭТАЛОН: Ровно в 15:00 слышен гимн и на этой небольшой Улочке. К патриотической акции присоединились трое скрипачей. Ребята специально приехали во Львов из Дрогобыча.
F16: Ровно в 15:00 слышен гимн и на этой небольшой Улочке. К патриотической акции присоединились трое скрипачей. Ребята специально при

In [56]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

root = Path("/content")

names = [
    "soup.yaml",
    "train.log",
    "gpu_train.csv",
    "merge.log",
    "quantization.log",
    "export_f16.log",
    "baseline_gec.json",
    "lora_gec_results.json",
    "merged_gec_results.json",
    "q4_gec_results.json",
    "f16_gec_results.json",
    "adapter_proof.json",
    "report.json",
    "soup_qwen_gec_output/soup_run.json",
    "llama_server.log",
    "llama_server_f16.log",
]

archive_path = root / "soup_final_evidence.zip"

with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for name in names:
        path = root / name

        if path.is_file():
            archive.write(path, name)
            print("OK:", name)
        else:
            print("НЕТ:", name)

print("\nАрхив создан:", archive_path)
print("Размер:", round(archive_path.stat().st_size / 1024**2, 2), "МБ")

files.download(str(archive_path))

OK: soup.yaml
OK: train.log
OK: gpu_train.csv
OK: merge.log
OK: quantization.log
OK: export_f16.log
OK: baseline_gec.json
OK: lora_gec_results.json
OK: merged_gec_results.json
OK: q4_gec_results.json
OK: f16_gec_results.json
OK: adapter_proof.json
OK: report.json
OK: soup_qwen_gec_output/soup_run.json
OK: llama_server.log
OK: llama_server_f16.log

Архив создан: /content/soup_final_evidence.zip
Размер: 0.05 МБ


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>